# GLiNER for entity resolution

**The thing to get straight first:** GLiNER does *not* do entity resolution. It does zero-shot **NER** — you hand it text and a list of label strings in plain English, it hands back spans. It will tell you that `"Red Bull GmbH"` is a `company`. It will *not* tell you that `"Red Bull GmbH"`, `"Red Bull North America, Inc."` and `"RedBull"` are the same company.

Entity resolution is the step *after*. So GLiNER slots into the pipeline in `semantic_entity_resolution_cell.py` as a drop-in replacement for the brittle regex `extract()` — and it is a big upgrade there, because the hard part of resolving these affiliation / COI / funding strings is that the mentions come out dirty. Clean mentions in, easy clustering out.

```
                 ┌──────────────────────────────── this notebook ────────────────────────────────┐
raw cell text →  1. EXTRACT  →  2. NORMALIZE  →  3. EMBED  →  4. BLOCK  →  5. SCORE  →  6. CLUSTER  →  canonical map
                    GLiNER       suffix strip     MiniLM       ANN/kNN    hybrid cos     union-find
                 └── replaces the regex ──┘       └──────── this part is already in your .py ────────┘
```

Why the extract step is worth replacing:

| input | regex `extract()` | GLiNER |
|---|---|---|
| `Juul Labs, Inc., San Francisco, CA, 94107, USA.` | `juul labs` + `san francisco` + junk | `Juul Labs, Inc.` → `company` |
| `Dr. Smith is a paid consultant for Red Bull and received honoraria from Monster Beverage Corp.` | splits on `and`, mangles both | `Red Bull` → `company`, `Monster Beverage Corp.` → `company` |
| `Funded by the National Institutes of Health (R01DA12345).` | `national institutes of health` (after grant-number regex) | `National Institutes of Health` → `funding agency` |

GLiNER also gives you a **type label** per span for free, which you can use to forbid cross-type merges in step 6 (never fuse a `university` into a `company`).

## 1. Install

`gliner` pulls torch + transformers. First run downloads model weights (~200 MB for medium, ~500 MB for large) into your HF cache.

In [15]:
import sys
!{sys.executable} -m pip install -q gliner sentence-transformers scikit-learn pandas rapidfuzz

You should consider upgrading via the '/Users/aisvarya/.pyenv/versions/3.10.3/bin/python -m pip install --upgrade pip' command.


## 2. Load the model

Model picks, all from the GLiNER hub. Start with **medium-v2.1** — it is the accuracy/speed sweet spot for org names.

| model | params | notes |
|---|---|---|
| `urchade/gliner_small-v2.1` | 166M | fast, use if you're iterating on labels |
| `urchade/gliner_medium-v2.1` | 209M | **default pick** |
| `urchade/gliner_large-v2.1` | 459M | best recall on messy multi-clause COI text |
| `urchade/gliner_multi-v2.1` | 209M | multilingual, for non-English affiliations |

Runs fine on CPU (~10–30 docs/sec small, ~3–8/sec large). On Apple silicon pass `map_location="mps"`.

In [16]:
from gliner import GLiNER
import torch

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
model = model.to(DEVICE).eval()

device: mps


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

## 3. Smoke test

`predict_entities(text, labels, threshold)` → list of `{text, label, score, start, end}`.

In [17]:
LABELS = ["company", "university", "research institute", "hospital", "funding agency", "government agency"]

samples = ['Juul Labs, Inc., United States.',
       'Juul Labs, Inc., United States. Electronic address: nicholas.goldenson@juul.com.',
       'Juul Labs, Inc.', 'Juul Labs, Inc., San Francisco, USA.',
       'JUUL Labs, Inc, Washington, DC, USA.',
       'Juul Labs, Inc, Washington, DC, United States.',
       'Regulatory Sciences, Juul Labs, Inc., Washington, DC 20004, USA.',
       'Department of Regulatory Sciences, Juul Labs, Inc., Washington, DC 20004, USA.',
       'Regulatory Sciences, Juul Labs, Inc., Washington DC 20004, USA.',
       'Juul Labs, Inc, Washington, DC, USA.',
       'Juul Labs Inc., San Francisco, CA, USA.',
       'Juul Labs, Inc., Washington, DC, USA.',
       'Juul Labs, Inc. Washington, DC, USA.',
       'JUUL Labs, Inc, Washington, District of Columbia, USA.',
       'Juul Labs, Inc, United States. Electronic address: Nicholas.Goldenson@Juul.com.',
       'Juul Labs, Inc, United States.',
       'Behavioral and Clinical Sciences, Juul Labs, Inc, Washington, DC, USA.',
       'JUUL Labs, 1000 F Street NW, Washington, DC 20004, USA.',
       'JUUL Labs, Inc., Washington, District of Columbia, USA.',
       'Juul Labs Inc., Washington, DC, United States.',
       'Behavioral and Clinical Sciences, Juul Labs, Inc., Washington, DC, USA.',
       'Population and Clinical Sciences, Juul Labs, Inc., Washington, DC, USA.',
       'Regulatory Sciences, Juul Labs, Inc., Washington, DC, USA.',
       'Juul Labs, Inc., Washington, DC, United States.',
       'Juul Labs, Inc, Washington DC, USA. Electronic address: Guy.Lalonde@JUUL.com.',
       'Juul Labs, Inc, Washington DC, USA.',
       'JUUL Labs, Inc., Washington, DC, USA.',
       'JUUL Labs, Inc., Washington, DC, USA. tengjiaochen@gmail.com.',
       'Juul Labs, Inc., Washington, DC, USA. nate.holt@juul.com.',
       'JUUL Labs, Inc, Washington, DC, United States.',
       'JUUL Labs, Inc, Washington, DC USA.',
       'Regulatory Science, JUUL Labs Inc., 1000 F Street NW, Washington D.C., 20004, USA.',
       'Juul Labs, Inc., San Francisco, CA, 94107, USA. Electronic address: guy.lalonde@juul.com.',
       'Enthalpy Analytical, Durham, NC. 27713, USA. Electronic address: gene.gillman@juul.com.',
       'Juul Labs, Inc., San Francisco, CA, 94107, USA. Electronic address: jenny.yao@juul.com.',
       'Juul Labs, Inc., San Francisco, CA, 94107, USA. Electronic address: kubilay.demir@juul.com.',
       'Juul Labs, Inc., San Francisco, CA, 94107, USA. Electronic address: michael.oldham@juul.com.',
       'JUUL Labs Inc, Washington, DC, USA.',
       'JUUL Labs Inc, Washington, DC, USA. Electronic address: tengjiao.chen@juul.com.',
       'Juul Labs, Inc, 1000 F Street NW, Suite 800, Washington, D.C, 20004, USA.',
       'Juul Labs, Inc, 1000 F Street NW, Suite 800, Washington, D.C, 20004, USA. nicholas.goldenson@juul.com.',
       'Juul Labs, Inc., USA.',
       'Juul Labs, Inc., USA. Electronic address: nicholas.goldenson@juul.com.',
       'Juul Laboratories, Inc., Washington, D.C. 20004, United States.',
       'Product Stewardship, JUUL Labs, Washington, DC, United States of America.',
       'Scientific Affairs, JUUL Labs, Washington, DC, United States of America.',
       'Regulatory Sciences, JUUL Labs, Washington, DC, United States of America.',
       'Modeling and Simulations Scientist, Juul Labs Inc, Washington, DC, United States.',
       'Director of Data Science and Analytics, Juul Labs Inc, Washington, DC, United States.',
       'Vice President Data, Juul Labs Inc, Washington, DC, United States.',
       'Director of Health Economics and Policy Research, Juul Labs Inc, Washington, DC, United States.',
       'Research Economist, Juul Labs Inc, Washington, DC, United States.',
       'Director of Regulatory Product and Trade Strategy, Juul Labs Inc, Washington, DC, United States.',
       'Senior Manager of Strategy, Juul Labs Inc, Washington, DC, United States.',
       'Vice President of Regulatory Engagement, Juul Labs Inc, Washington, DC, United States.',
       'Juul Labs Inc, Director, Behavioral Affairs, Washington, DC, United States.',
       'Associate, Behavioral Science Research, Juul Labs Inc, Washington, DC, United States.',
       'JUUL Labs, Inc., 560 20th Street, San Francisco, USA.',
       'JUUL Labs, Inc., 560 20th Street, San Francisco, USA. Michael.Oldham@juul.com.']

for s in samples:
    print(s)
    for e in model.predict_entities(s, LABELS, threshold=0.5):
        print(f"   {e['text']:<45} {e['label']:<18} {e['score']:.2f}")
    print()

Juul Labs, Inc., United States.
   Juul Labs                                     company            0.75

Juul Labs, Inc., United States. Electronic address: nicholas.goldenson@juul.com.
   Juul Labs                                     company            0.93

Juul Labs, Inc.
   Juul Labs, Inc.                               company            0.56

Juul Labs, Inc., San Francisco, USA.
   Juul Labs, Inc.                               company            0.81

JUUL Labs, Inc, Washington, DC, USA.
   JUUL Labs, Inc                                company            0.81

Juul Labs, Inc, Washington, DC, United States.
   Juul Labs                                     company            0.84

Regulatory Sciences, Juul Labs, Inc., Washington, DC 20004, USA.
   Regulatory Sciences                           research institute 0.61
   Juul Labs                                     company            0.91

Department of Regulatory Sciences, Juul Labs, Inc., Washington, DC 20004, USA.
   Department o

## 4. Tune the labels — this is the actual work

GLiNER is **very** sensitive to label wording. The labels are a prompt, not a fixed taxonomy. Rules that hold up in practice:

- **Lowercase, singular, generic.** `"company"` beats `"Company"` beats `"Corporations and businesses"`.
- **Split types you want to treat differently downstream.** `"funding agency"` vs `"company"` lets you keep NIH grants apart from industry money — the whole point of a COI graph.
- **Add *sink* labels for things you want to throw away.** This is the counterintuitive one and it's the single biggest precision win. GLiNER must assign *some* label from your list to every span it finds — if there's no right bucket, it picks the least-wrong one. With no `country` label, `United States of America` gets forced into `government agency`. Add `"country"` and `"city"`, then discard those spans after extraction. **Verified on this corpus:** `United States of America` moves from `government agency` (0.68) to `country` (0.94).
- **Threshold is a real knob, but it won't fix a bad label set.** 0.3 for recall (you're going to cluster anyway, and a stray mention usually lands in a singleton cluster you can drop), 0.7 when precision matters more. Tempting here to just raise it — don't: the department false positives score 0.77–0.94, *above* real entities like `American Beverage Association` (0.64). Any threshold that kills the noise kills real funders first.

### Sink labels alone don't fix departments — you need §6's positional rule

Corporate divisions (`Behavioral Science Research`, `Scientific Affairs`, `Regulatory Sciences`) are genuinely ambiguous in isolation; `"Behavioral Science Research"` really does look like a research org. Adding a `department` sink is not enough, because `research institute` outcompetes it (0.92 vs 0.87). Measured on 10 affiliation strings:

| approach | clean |
|---|---|
| current label set | 4/10 |
| + `department` sink | 5/10 |
| + `country`/`city` sinks too | 7/10 |
| drop `research institute` entirely | 8/10 — but then `RTI International` misfires to `company` (0.50) |
| **sinks + positional rule (§6)** | **10/10** |

The disambiguating signal isn't semantic, it's **positional** — see §6. Run the sweep below on your own strings before committing.

In [ ]:
import pandas as pd

LABEL_SETS = {
    "coarse":     ["organization"],
    "typed":      ["company", "university", "funding agency"],
    "typed+":     ["company", "university", "research institute", "hospital",
                   "funding agency", "government agency"],
    "typed+role": ["company", "university", "funding agency", "trade association",
                   "beverage company"],
}

rows = []
for name, labels in LABEL_SETS.items():
    for s in samples:
        ents = model.predict_entities(s, labels, threshold=0.4)
        rows.append({
            "label_set": name,
            "text": s[:55] + "…",
            "found": ", ".join(f"{e['text']}[{e['label']}]" for e in ents) or "—",
        })
pd.DataFrame(rows).style.set_properties(**{"text-align": "left"})

## 5. Pull the real text out of Postgres

Same `.env` connection as `explore-db.ipynb`.

In [18]:
import os
from pathlib import Path
from sqlalchemy import create_engine, text
import pandas as pd

# Anchor on the project root, not the kernel's cwd -- this notebook lives one level
# down in entity_dedup_experiments/, and the paths inside .env (DB_SSL_CA) are
# written relative to the root. Walking up makes it work from either directory.
def project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for d in (start, *start.parents):
        if (d / ".env").exists() and (d / "package.json").exists():
            return d
    raise FileNotFoundError("project root not found (looked for .env + package.json)")

ROOT = project_root()

def load_env(path):
    env = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env(ROOT / ".env")

ca_path = Path(env.get("DB_SSL_CA", "global-bundle.pem"))
if not ca_path.is_absolute():
    ca_path = ROOT / ca_path            # .env paths are root-relative
ca_path = ca_path.resolve()
if not ca_path.exists():
    raise FileNotFoundError(f"CA bundle missing: {ca_path}")
print(f"root={ROOT.name}  ca={ca_path.name}")

engine = create_engine("postgresql+psycopg2://", connect_args={
    "host": env["DB_HOST"], "port": int(env.get("DB_PORT", 5432)),
    "dbname": env["DB_NAME"], "user": env["DB_USER"], "password": env["DB_PASSWORD"],
    "sslmode": "verify-full", "sslrootcert": str(ca_path),
})

# One long table of (source_row, field, raw_text). GLiNER wants raw text, not
# the pre-split institution names -- it does the splitting better than the ETL did.
docs = pd.read_sql(text("""
    SELECT article_id::text AS row_id, 'coi' AS field, "COI_raw_text" AS raw
      FROM author_coi WHERE NULLIF(btrim("COI_raw_text"), '') IS NOT NULL
    UNION ALL
    SELECT af.article_id::text, 'affiliation', i.name
      FROM author_affiliation af JOIN institution i ON i.institution_id = af.institution_id
    UNION ALL
    SELECT fu.article_id::text, 'funding', i.name
      FROM article_funding fu JOIN institution i ON i.institution_id = fu.institution_id
"""), engine).drop_duplicates()

print(f"{len(docs)} raw strings")
docs.head()

root=pubmed-semantic-interactive copy  ca=global-bundle.pem
77258 raw strings


,row_id,field,raw
0,3,coi,The authors declare no competing financial int...
1,12568,coi,The authors declare that they have no known co...
10,7,coi,The mybluTM devices used in this study were ma...
13,10299,coi,None.
20,10300,coi,The authors declared no conflict of interest.


## 6. EXTRACT — batch GLiNER over everything

Four things that bite here:

- **Context window.** GLiNER truncates past ~384 tokens. COI statements blow through that regularly, so chunk on sentence boundaries first or you silently lose every entity in the tail.
- **Batching.** `batch_predict_entities` is several times faster than looping `predict_entities`. Keep batches at 8–32 on CPU.
- **Sink labels** (`country`, `city`, `corporate department`, `academic department`) are extracted and then dropped. They exist to absorb spans that would otherwise be forced into a real label.
- **The positional rule.** PubMed affiliations are ordered `department, organization, city, state, country`. So in an affiliation, any org-ish span starting *before* the first `company` span is a division of that company, not an organization — that's what turns `Behavioral Science Research[university], Juul Labs Inc[company]` into just `Juul Labs Inc[company]`.

> ⚠️ **Scope the positional rule to `field == "affiliation"` only.** COI and funding prose has no such ordering, and applying it there silently deletes real entities. Verified: on *"This study was supported by the American Beverage Association and Coca-Cola."* the rule drops `American Beverage Association` because it precedes `Coca-Cola`. The `if field == "affiliation"` guard below is load-bearing.

In [ ]:
import re
from tqdm.auto import tqdm

# Entities we keep.
KEEP_LABELS = ["company", "university", "research institute", "hospital",
               "funding agency", "government agency"]
# SINK labels: extracted, then thrown away. They exist so GLiNER has a correct
# bucket for these spans instead of forcing them into a label we keep.
SINK_LABELS = ["corporate department", "academic department", "country", "city"]

LABELS = KEEP_LABELS + SINK_LABELS
SINK = set(SINK_LABELS)

THRESHOLD = 0.4
BATCH = 16
MAX_CHARS = 1200          # ~= 384 tokens, GLiNER's context limit

def chunk(s, max_chars=MAX_CHARS):
    """Split on sentence boundaries so nothing past the context window is lost."""
    s = str(s).strip()
    if len(s) <= max_chars:
        return [s]
    out, cur = [], ""
    for sent in re.split(r"(?<=[.;])\s+", s):
        if len(cur) + len(sent) > max_chars and cur:
            out.append(cur); cur = sent
        else:
            cur = f"{cur} {sent}".strip()
    if cur:
        out.append(cur)
    return out

def drop_subunits(ents):
    """A PubMed affiliation runs: department, organization, city, state, country.
    So any span starting BEFORE the first `company` span is that company's internal
    division, not an organization in its own right.

    AFFILIATION FIELD ONLY. COI/funding prose has no such ordering -- applying this
    there deletes real entities (verified: it drops `American Beverage Association`
    from "...supported by the American Beverage Association and Coca-Cola.").
    """
    comps = [e for e in ents if e["label"] == "company"]
    if not comps:
        return ents
    first = min(e["start"] for e in comps)
    return [e for e in ents if e["label"] == "company" or e["start"] > first]

# Flatten to (row_id, field, chunk_text) so batching is uniform.
units = [(r.row_id, r.field, c) for r in docs.itertuples() for c in chunk(r.raw)]
print(f"{len(units)} chunks")

mentions = []
for i in tqdm(range(0, len(units), BATCH)):
    batch = units[i:i + BATCH]
    preds = model.batch_predict_entities([u[2] for u in batch], LABELS, threshold=THRESHOLD)
    for (row_id, field, txt), ents in zip(batch, preds):
        ents = [e for e in ents if e["label"] not in SINK]   # discard sink spans
        if field == "affiliation":                            # <- guard is load-bearing
            ents = drop_subunits(ents)
        for e in ents:
            mentions.append({
                "row_id": row_id, "field": field, "raw": txt,
                "surface": e["text"], "type": e["label"], "score": e["score"],
            })

men = pd.DataFrame(mentions)
print(f"{len(men)} mentions, {men.surface.nunique()} distinct surface forms")
men.type.value_counts()

## 7. NORMALIZE

Cheap and high-leverage. Strip legal suffixes and punctuation so `"Red Bull GmbH"` and `"Red Bull North America, Inc."` at least start from the same stem. Do **not** skip this and expect the embedder to cover for it — embeddings treat `Inc.` as signal.

In [ ]:
LEGAL = r"\b(inc|llc|ltd|limited|corp|corporation|co|company|gmbh|ag|sa|nv|bv|plc|lp|llp|"
LEGAL += r"pty|kk|srl|spa|as|ab|oy|aps|holdings?|group|international|worldwide|"
LEGAL += r"north america|usa|u s a|united states)\b"

def normalize(s):
    s = str(s).lower()
    s = re.sub(r"[‘’“”]", "'", s)
    s = re.sub(r"\(.*?\)", " ", s)              # (R01DA12345), (the "Company")
    s = re.sub(r"[^a-z0-9&' ]", " ", s)
    s = re.sub(LEGAL, " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

men["norm"] = men.surface.map(normalize)
men = men[men.norm.str.len() >= 3].copy()

# Collapse to unique (norm, type) -- cluster the vocabulary, not every occurrence.
vocab = (men.groupby(["norm", "type"])
            .agg(n=("surface", "size"), surfaces=("surface", lambda s: sorted(set(s))))
            .reset_index()
            .sort_values("n", ascending=False))
print(f"{len(vocab)} unique (normalized, type) pairs")
vocab.head(25)

## 8. EMBED + BLOCK + SCORE + CLUSTER

The part GLiNER doesn't do.

**Use a hybrid score, not raw cosine.** Sentence embeddings are trained on sentences; on 2–4 word org names they cheerfully rate `"Red Bull"` and `"Monster Beverage"` as similar — both are energy drink companies, which is exactly the semantic relation you *don't* want here. Mixing in a character-level ratio fixes it:

```
score = 0.6 · cosine(embedding)  +  0.4 · token_set_ratio(strings)
```

**Guard against transitive chaining.** Union-find merges `A~B` and `B~C` into one cluster even when `A` and `C` are unrelated, so one bad edge can swallow half your vocabulary. Two guards below: require a shared token on every edge, and forbid cross-type merges.

In [ ]:
import collections
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from rapidfuzz import fuzz

SIM_THRESHOLD = 0.72
K = 15
W_COS, W_FUZZ = 0.6, 0.4

names = vocab.norm.tolist()
types = vocab.type.tolist()

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
X = embedder.encode(names, normalize_embeddings=True, show_progress_bar=True)

# BLOCK: kNN instead of all-pairs -- O(n log n) not O(n^2).
nn = NearestNeighbors(n_neighbors=min(K, len(names)), metric="cosine").fit(X)
dist, idx = nn.kneighbors(X)

parent = list(range(len(names)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x

tokens = [set(n.split()) for n in names]
edges = []
for i in range(len(names)):
    for d, j in zip(dist[i], idx[i]):
        if i >= j:
            continue
        if types[i] != types[j]:                       # guard: no cross-type merges
            continue
        if not (tokens[i] & tokens[j]):                # guard: no chaining on pure semantics
            continue
        score = W_COS * (1.0 - d) + W_FUZZ * (fuzz.token_set_ratio(names[i], names[j]) / 100)
        if score >= SIM_THRESHOLD:
            edges.append((score, i, j))
            parent[find(i)] = find(j)

print(f"{len(edges)} merge edges")

groups = collections.defaultdict(list)
for i in range(len(names)):
    groups[find(i)].append(i)
print(f"{len(names)} mentions -> {len(groups)} clusters")

## 9. Canonical labels + the resolved map

Canonical form = the most frequent surface form in the cluster, tie-broken by shortest. Frequency beats "longest/most official" here — the common form is the one your readers will recognise.

In [ ]:
canon_rows = []
for members in groups.values():
    cnt = collections.Counter()
    for i in members:
        for s in vocab.iloc[i].surfaces:
            cnt[s] += vocab.iloc[i].n
    canonical = sorted(cnt.items(), key=lambda kv: (-kv[1], len(kv[0])))[0][0]
    for i in members:
        canon_rows.append({
            "canonical": canonical,
            "type": types[i],
            "norm": names[i],
            "surfaces": "; ".join(vocab.iloc[i].surfaces),
            "n": int(vocab.iloc[i].n),
            "cluster_size": len(members),
        })

canon = pd.DataFrame(canon_rows).sort_values(["cluster_size", "canonical", "n"],
                                             ascending=[False, True, False])

# The clusters that actually merged something -- these are what you inspect.
merged = canon[canon.cluster_size > 1]
print(f"{merged.canonical.nunique()} multi-member clusters")
merged.head(40)

In [ ]:
# Spot-check the entities this project cares about.
for probe in ["red bull", "monster", "coca cola", "pepsi", "national institutes of health",
              "american beverage", "juul", "altria"]:
    hits = canon[canon.norm.str.contains(probe, na=False)]
    if len(hits):
        print(f"--- {probe} ---")
        for c, grp in hits.groupby("canonical"):
            print(f"  {c}  <-  {', '.join(sorted(set(grp.norm)))}")
        print()

In [ ]:
# mention-level table: every original occurrence, now carrying a canonical id
resolved = men.merge(canon[["norm", "type", "canonical"]], on=["norm", "type"], how="left")
resolved.to_csv("gliner_resolved_entities.csv", index=False)
print(f"wrote gliner_resolved_entities.csv  ({len(resolved)} rows, "
      f"{resolved.canonical.nunique()} canonical entities)")
resolved.head(20)

## 10. Tuning, and where this breaks

**Knobs, in the order they're worth touching:**

1. `LABELS` — biggest effect by far, and **sink labels are most of that effect**. Re-run §4 whenever you change them.
2. `THRESHOLD` (GLiNER) — low (0.3) favours recall; junk mostly lands in singleton clusters you can drop with `cluster_size == 1 & n == 1`.
3. `SIM_THRESHOLD` — raise it if unrelated orgs are fusing, lower it if `"Red Bull"` and `"Red Bull GmbH"` stay apart. Watch a fixed set of known pairs while you move it rather than eyeballing the whole table.
4. `W_COS` / `W_FUZZ` — push toward fuzz for org names, toward cosine for descriptive strings ("the NIH" vs "National Institutes of Health" needs semantics, and neither shares a token — see the known gap below).

**Known gaps in this pipeline:**

- **Acronyms.** `"NIH"` and `"National Institutes of Health"` share no token and no characters, so both guards block the merge. Fix with an explicit alias table — a hand-written dict of ~50 aliases will beat any amount of threshold tuning for this project's long tail.
- **Parent/subsidiary.** Is `"Red Bull North America"` the same entity as `"Red Bull GmbH"`? For a COI graph, usually yes; for a legal analysis, no. That's a judgement call the model can't make for you — decide it, then encode it in the alias table.
- **No relation type.** GLiNER gives you `Red Bull → company` but not *consultant for* vs *funded by*. If the COI graph needs that edge type, look at [GLiREL](https://github.com/jackboyla/GLiREL) (same zero-shot idea, for relations) or `knowledgator/gliner-multitask-large-v0.5`.
- **Common-noun false positives in COI prose.** GLiNER will tag bare words like `researchers` as `research institute` (seen at 0.4–0.6 on real COI text). The positional rule can't help — it's not an affiliation. Cheapest fix is a stopword set of ~20 common nouns (`researchers`, `authors`, `investigators`, `study`, `participants`) filtered right after extraction.
- **No ground truth.** Hand-label 100 mentions once and keep them in a CSV. Pairwise precision/recall against those 100 turns threshold tuning from vibes into a number, and it's an afternoon of work.

**Before wiring this into the app:** the existing `institution` table already has IDs. Map `canonical` → `institution_id` and write the crosswalk to a new table rather than mutating `institution` in place — that keeps the resolution reversible when you change the thresholds, which you will.